# Session 6 — LangGraph: State, Nodes, Edges & Durable Workflows

For the last three sessions you have used `create_agent` — one call that gives an LLM tools, memory, and a loop. That convenience hides a real machine underneath: a **graph**. Today we open it up.

You will build graphs by hand so the abstraction stops being magic. Then you will see the two things a plain agent cannot easily give you: **explicit control over the path** (some steps must run in a fixed order) and **durable state** (a case that survives a restart, resumes after approval, and remembers a customer across conversations).

**The Northstar thread:** it is Launch Week and one message arrives — *"I was charged twice, I got a password-reset email I didn't request, and the service was down during my demo."* One overloaded agent fumbles it. By the end of this notebook you will have the vocabulary to design something better.

By the end you can:

- name and use every LangGraph building block — **state, nodes, edges, conditional edges, compile**;
- explain **why** and **when** to reach for a graph instead of a single agent, and where **workflows**, **agents**, and **multi-agent** systems sit on one spectrum;
- make a graph **remember** — with a **checkpointer** (one conversation) and a **store** (across conversations).

Reference for the whole session: [LangGraph overview](https://docs.langchain.com/oss/python/langgraph/overview).


## 1. Setup

🎯 **Purpose:** install LangGraph and confirm the Gemini key is available. We reuse the same model and `.env` as Sessions 3–5.

> If an import still uses an older version after the first install, restart the kernel and run the imports again.


In [ ]:
%pip install -q -U langgraph langchain langchain-google-genai python-dotenv

In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)  # reads .env in this folder; never hard-code keys

import os

# One place to swap the model later — same idea as every prior session.
MODEL = "google_genai:gemini-3.1-flash-lite"

print("GEMINI_API_KEY:", "set" if os.getenv("GEMINI_API_KEY") else "MISSING")
print("Model:", MODEL)
# Never print the key itself.

## 2. Why LangGraph? — thinking in graphs

`create_agent` is perfect when the model can be trusted to *decide its own steps*: look something up, maybe call another tool, then answer. But some jobs are not like that.

Northstar's refund process has **rules**: verify identity **before** discussing eligibility, get a human approval **before** money moves. "Give one agent every tool and hope it picks the right order" is not enforcement — it is a suggestion. And when the process must pause for a manager overnight and resume tomorrow, a single in-memory agent loop has nowhere to stand.

LangGraph is the **orchestration runtime** underneath LangChain agents. You describe your process as a **graph**: boxes (steps) and arrows (what can run next), plus a shared **state** every step reads and writes. Because the runtime executes that graph step by step, it can do things a plain function cannot: **pause, resume, recover, remember, stream progress, and show you exactly where it is**.

💡 **Real-life:** a plain agent is a talented employee improvising. A LangGraph graph is the **written procedure** that employee follows — you can see every step, insert an approval, and pick up where you left off if the office closes.


### 2.1 Workflows vs. Agents — one spectrum

Two ways to build with LLMs, and everything in between:

- **Workflow** — *you* decide the path. Steps run in a predetermined order. Predictable, testable, auditable.
- **Agent** — the *model* decides the path. It chooses tools and when to stop. Flexible, but the route varies run to run.

| | Workflow | Agent |
|---|---|---|
| Who chooses the next step? | You (the graph's edges) | The model |
| Order guarantees | Strong — encoded in code | Weak — a hint in the prompt |
| Best for | Regulated / ordered / branching processes | Open-ended tasks, "figure it out" |
| Northstar example | Refund: verify → check → approve → pay | "Answer this support question" |
| In LangGraph | `StateGraph` you wire yourself | `create_agent` (a prebuilt graph) |

The important idea: **`create_agent` is itself a LangGraph graph.** Same building blocks. Once you can build a graph, you can mix deterministic steps with model-driven ones in the *same* system.

Reference: [Workflows and agents](https://docs.langchain.com/oss/python/langgraph/workflows-agents).


### 2.2 How to think in LangGraph — 5 steps

Whenever you design a graph, walk this path (from the docs' *Thinking in LangGraph*):

1. **Map the process into discrete steps.** Each becomes a **node**.
2. **Decide what each step is:** an LLM call, a data lookup, an external action, or a human pause.
3. **Design the state** — the shared notebook every node reads and writes. Store **raw data, not formatted prompt strings**; build prompts inside nodes on demand.
4. **Write the nodes** — plain functions: take state, do one thing, return the fields that changed.
5. **Wire the edges** — connect the steps, adding branches where a decision picks the path.

We now do exactly this, smallest possible graph first.

Reference: [Thinking in LangGraph](https://docs.langchain.com/oss/python/langgraph/thinking-in-langgraph).


## 3. The four building blocks

Every LangGraph graph is made of exactly four things:

| Block | What it is | One-line job |
|---|---|---|
| **State** | A `TypedDict` schema | The shared data passed between steps |
| **Node** | A Python function `state → updates` | Does one unit of work |
| **Edge** | A connection between nodes | Says what can run next |
| **Compile** | `builder.compile()` | Validates and produces the runnable graph |

💡 **Real-life — an assembly line:** the **state** is the item moving down the belt with its worksheet attached; each **node** is a station that does one task and writes on the worksheet; the **edges** are the conveyor that carries it to the next station; **compile** is switching the line on.

We will build this tiny support graph: `START → classify → answer → END`.


### 3.1 State — the shared notebook

🎯 **Purpose:** define what data flows through the graph. State is a `TypedDict`: a dictionary with named, typed fields. Every node receives the whole state and returns a partial update.

💡 **Real-life:** a **shared clipboard**. Each worker reads it, writes their part, and passes it on — nobody keeps a private copy.


In [ ]:
from typing_extensions import TypedDict


# The schema of everything that flows through our graph.
class State(TypedDict):
    message: str    # the incoming customer message (input)
    category: str   # filled in by the classify node
    answer: str     # filled in by the answer node


# It is just a dict shape — no magic. These are valid states:
print(State.__annotations__)
example = {"message": "I was charged twice", "category": "", "answer": ""}
print(example)

### 3.2 Nodes — functions that return updates

🎯 **Purpose:** a node is an ordinary function. It takes the current `state` and returns **only the fields it wants to change** — LangGraph merges that update back into the shared state for you.

💡 **Real-life:** a worker at a station writes *their* line on the clipboard and hands it on. They don't rewrite the whole sheet — just their part.


In [ ]:
def classify(state: State) -> dict:
    """Look at the message and decide a category."""
    text = state["message"].lower()
    category = "security" if "password" in text else "billing"
    return {"category": category}   # only the field we changed


def answer(state: State) -> dict:
    """Use the category to produce a reply."""
    return {"answer": f"Routed to {state['category']} support."}


# A node is just a function — call it directly to demystify it:
print(classify({"message": "I received an unknown password reset", "category": "", "answer": ""}))

### 3.3 Edges — wire the nodes into a graph

🎯 **Purpose:** now connect the steps. `START` and `END` are the built-in entry and exit points. `add_edge(a, b)` means "after `a`, always go to `b`." Then `compile()` turns the blueprint into something runnable with `.invoke()`.

💡 **Real-life:** the conveyor belt between stations, and flipping the power switch on the line.


In [ ]:
from langgraph.graph import StateGraph, START, END

graph = (
    StateGraph(State)             # 1. a builder tied to our state schema
    .add_node("classify", classify)   # 2. register the steps
    .add_node("answer", answer)
    .add_edge(START, "classify")      # 3. entry -> classify
    .add_edge("classify", "answer")   #    classify -> answer
    .add_edge("answer", END)          #    answer -> exit
    .compile()                        # 4. build the runnable graph
)

result = graph.invoke({"message": "I received an unknown password reset"})
print(result)
# Note: we only passed 'message'. The nodes filled in 'category' and 'answer'.

### 3.4 See the graph

🎯 **Purpose:** a graph is a picture, so look at it. `draw_mermaid()` prints a text diagram with no extra dependencies. (`draw_mermaid_png()` renders an image but calls an online service — handy in class, skip it offline.)


In [ ]:
# Text diagram — paste into any Mermaid viewer, or just read the arrows.
print(graph.get_graph().draw_mermaid())

# For an inline image (needs internet), you could instead run:
# from IPython.display import Image, display
# display(Image(graph.get_graph().draw_mermaid_png()))

## 4. Conditional edges — let the graph decide where to go

So far every arrow was fixed. Real support isn't: a security problem, a billing problem, and a general question should go to **different** desks. A **conditional edge** runs a small function that looks at the state and returns the **name of the next node**.

💡 **Real-life:** a **triage nurse**. One intake step reads the situation and sends each patient down a different corridor — orthopedics, cardiology, or general.


### 4.1 Anatomy of a conditional edge

Three parts:

1. A **router function** `state → next node name` (return a plain string, or `END`).
2. `add_conditional_edges("source_node", router, [list of possible destinations])`.
3. Each destination node still needs its own edge onward (usually to `END`).

The router **does not do the work** — it only decides the direction. Keep it small and cheap.


In [ ]:
from typing import Literal


class RouteState(TypedDict):
    message: str
    category: str
    answer: str


def classify_ticket(state: RouteState) -> dict:
    """Decide which desk should handle this message."""
    text = state["message"].lower()
    if "password" in text or "hacked" in text:
        category = "security"
    elif "charge" in text or "refund" in text or "invoice" in text:
        category = "billing"
    else:
        category = "general"
    return {"category": category}


def route(state: RouteState) -> Literal["security", "billing", "general"]:
    """The conditional edge: pick the next node based on state."""
    return state["category"]


def security(state: RouteState) -> dict:
    return {"answer": "Security desk: let's secure your account first."}


def billing(state: RouteState) -> dict:
    return {"answer": "Billing desk: let's look at your charges."}


def general(state: RouteState) -> dict:
    return {"answer": "General desk: happy to help."}


router_graph = (
    StateGraph(RouteState)
    .add_node("classify", classify_ticket)
    .add_node("security", security)
    .add_node("billing", billing)
    .add_node("general", general)
    .add_edge(START, "classify")
    # classify -> (one of) security / billing / general, chosen by `route`
    .add_conditional_edges("classify", route, ["security", "billing", "general"])
    .add_edge("security", END)
    .add_edge("billing", END)
    .add_edge("general", END)
    .compile()
)

for msg in ["I was charged twice", "my account was hacked", "hello there"]:
    out = router_graph.invoke({"message": msg})
    print(f"{msg!r:28} -> {out['category']:9} | {out['answer']}")

In [ ]:
# The branch is visible in the diagram: classify now fans out to three nodes.
print(router_graph.get_graph().draw_mermaid())

### 4.2 Recap — the building blocks so far

| Block | Code | Job |
|---|---|---|
| State | `class S(TypedDict): ...` | Shared, typed data |
| Node | `def n(state) -> dict` | One unit of work; returns changed fields |
| Normal edge | `.add_edge("a", "b")` | Always go a → b |
| Conditional edge | `.add_conditional_edges("a", router, [...])` | Router picks the next node |
| Entry / exit | `START`, `END` | Where the graph begins and ends |
| Compile | `.compile()` | Produce the runnable graph |

That is the whole vocabulary. Everything else — agents, persistence, streaming — is built on these.

Reference: [Graph API quickstart](https://docs.langchain.com/oss/python/langgraph/quickstart).


## 5. Workflow, agent, multi-agent — in code

The graph we just built is a **workflow**: *we* decided the route. Now place it next to the other end of the spectrum — an **agent**, where the *model* decides — and then combine them.


### 5.1 A workflow: fixed steps in a fixed order

🎯 **Purpose:** a two-step LLM chain — draft, then polish. Each node is an LLM call; the **order is guaranteed by the edges**, not by a prompt. This is "prompt chaining" from the workflows guide.


In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(MODEL)


class ChainState(TypedDict):
    topic: str
    draft: str
    polished: str


def draft_node(state: ChainState) -> dict:
    msg = model.invoke(f"Write a one-sentence support tagline about {state['topic']}.")
    return {"draft": msg.text}


def polish_node(state: ChainState) -> dict:
    msg = model.invoke(f"Rewrite this to sound warmer, one sentence: {state['draft']}")
    return {"polished": msg.text}


chain = (
    StateGraph(ChainState)
    .add_node("draft", draft_node)
    .add_node("polish", polish_node)
    .add_edge(START, "draft")
    .add_edge("draft", "polish")   # polish ALWAYS follows draft
    .add_edge("polish", END)
    .compile()
)

out = chain.invoke({"topic": "fast refunds"})
print("draft   :", out["draft"])
print("polished:", out["polished"])

### 5.2 An agent: the model decides the steps

🎯 **Purpose:** `create_agent` (Sessions 3–5) is a **prebuilt LangGraph graph** whose nodes loop *model → tools → model* until the model is done. You don't wire the edges — the model chooses whether and when to call a tool.

💡 **Real-life:** the workflow is a **recipe** (same steps every time); the agent is a **chef** who tastes and decides what to do next.


In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool


@tool
def get_refund_days(plan: str) -> str:
    """Return how many business days a refund takes for a plan (Basic or Pro)."""
    return "3 business days" if plan.lower() == "pro" else "5 business days"


agent = create_agent(
    model=MODEL,
    tools=[get_refund_days],
    system_prompt="You are a Northstar billing assistant. Use tools for facts.",
)

res = agent.invoke({"messages": [{"role": "user", "content": "How long is a refund on the Pro plan?"}]})
print(res["messages"][-1].text)
# The model chose to call get_refund_days, read the result, then answered.

### 5.3 Multi-agent — specialists inside one system

🎯 **Purpose:** a single agent with *every* tool and a 40-line prompt gets overloaded — it retrieves the right policy but forgets the security concern, or starts a refund before verifying identity. A **multi-agent** system splits the work into focused specialists coordinated together.

The key move: **an agent can be wrapped as a tool for another agent.** A *supervisor* consults specialists, each of which sees only its own small toolset and context, then the supervisor writes one answer.

💡 **Real-life:** a **case manager** who asks a billing analyst and a security analyst for focused findings, then gives the customer one coherent plan — instead of five experts arguing at once.

Common patterns (named here, explored in depth next session): **subagents** (consult and report back), **handoffs** (transfer the conversation), **router** (classify then dispatch), **supervisor** (coordinate over several turns). Below is the smallest supervisor-consults-specialist example.


In [ ]:
# A focused specialist: small toolset, narrow instructions.
billing_specialist = create_agent(
    model=MODEL,
    tools=[get_refund_days],
    system_prompt="You are the billing specialist. Give a short, factual finding.",
)


# Wrap the specialist AGENT as a TOOL the supervisor can call.
@tool
def consult_billing(question: str) -> str:
    """Ask the billing specialist about charges, invoices, and refund timing."""
    res = billing_specialist.invoke({"messages": [{"role": "user", "content": question}]})
    return res["messages"][-1].text


supervisor = create_agent(
    model=MODEL,
    tools=[consult_billing],
    system_prompt=(
        "You are the Northstar case manager. Consult the relevant specialist, "
        "then give the customer one clear answer."
    ),
)

res = supervisor.invoke({"messages": [{"role": "user", "content": (
    "A Pro-plan customer asks how long their refund will take. Please help."
)}]})
print(res["messages"][-1].text)

## 6. Persistence — make the graph remember

Everything so far forgets instantly. Each `invoke` starts from nothing.

**The Northstar story:** Priya verifies her identity at 11:55. At noon the app is redeployed. At 2:00 she replies with an invoice ID. If the system starts from zero, it asks her to verify again — and loses her trust.

LangGraph has **two** kinds of memory, and they are not the same:

| | **Checkpointer** | **Store** |
|---|---|---|
| Saves | Full graph state, as snapshots | Your own key/value data |
| Scope | **One thread** (one conversation/case) | **Across threads** |
| Use for | Conversation memory, pause/resume, human approval, recovery | Preferences, durable facts, shared knowledge |
| Keyed by | `thread_id` | a namespace tuple |

💡 **Real-life:** a checkpointer is a **game save file** for one playthrough; a store is your **player profile** that carries across every playthrough. Most real apps use both.

Reference: [Persistence](https://docs.langchain.com/oss/python/langgraph/persistence).


### 6.1 Checkpointers — save state after every step

🎯 **Purpose:** compile with a **checkpointer** and every run is saved under a `thread_id`. Send follow-up messages on the **same** `thread_id` and the graph remembers the conversation. Use a **new** `thread_id` and it starts fresh.

💡 **Real-life:** **save points** in a video game — reach level nine, crash, resume near level nine, not the opening screen. The `thread_id` names the save slot.

`MessagesState` is a built-in state whose `messages` field automatically **appends** new messages (a "reducer") instead of overwriting — exactly what a conversation needs.


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import MessagesState

chat_model = init_chat_model(MODEL)


def call_model(state: MessagesState) -> dict:
    return {"messages": [chat_model.invoke(state["messages"])]}


chat_graph = (
    StateGraph(MessagesState)
    .add_node("model", call_model)
    .add_edge(START, "model")
    .add_edge("model", END)
    .compile(checkpointer=InMemorySaver())   # <-- the one line that adds memory
)

# thread_id is the memory key. Same id = same conversation.
thread_a = {"configurable": {"thread_id": "case-priya"}}

chat_graph.invoke({"messages": [{"role": "user", "content": "My invoice is INV-5013."}]}, thread_a)
reply = chat_graph.invoke({"messages": [{"role": "user", "content": "What was my invoice ID?"}]}, thread_a)
print("same thread :", reply["messages"][-1].text)

In [ ]:
# A DIFFERENT thread_id is a different conversation — no shared memory.
thread_b = {"configurable": {"thread_id": "case-sam"}}
reply_b = chat_graph.invoke({"messages": [{"role": "user", "content": "What was my invoice ID?"}]}, thread_b)
print("new thread  :", reply_b["messages"][-1].text)
# Teaching point: thread_id is NOT a user id. It names ONE case/conversation.

### 6.2 Inspect the saved state and its history

🎯 **Purpose:** because state is saved, you can look at it. `get_state(config)` returns the latest snapshot; `get_state_history(config)` returns every checkpoint (newest first). LangGraph saves one checkpoint per **super-step** — one "tick" of the graph — which is what makes pause/resume, recovery, and time-travel debugging possible.


In [ ]:
snapshot = chat_graph.get_state(thread_a)
print("state fields :", list(snapshot.values.keys()))
print("messages kept:", len(snapshot.values["messages"]))
print("next node(s) :", snapshot.next, "  (empty () means the run finished)")

print("\ncheckpoint history for thread 'case-priya' (newest first):")
for snap in chat_graph.get_state_history(thread_a):
    ckpt_id = snap.config["configurable"]["checkpoint_id"]
    print(f"  {ckpt_id[:12]}...  next={snap.next}")

### 6.3 Stores — remember across conversations

🎯 **Purpose:** a checkpointer keeps *one* conversation. A **store** keeps facts that should outlive every conversation — a preferred language, a communication channel — organized under a **namespace** (a tuple you design).

💡 **Real-life:** a **hotel** doesn't reuse last night's room-service order, but it *does* remember you need a wheelchair-accessible room — every stay. Northstar shouldn't merge Priya's tickets into one thread, but it can remember she prefers email in Bengali.


In [ ]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

# Namespace = a tuple you design. Put deliberate structure in it: it doubles as
# an authorization boundary (who is allowed to read this slice?).
namespace = ("northstar", "customer", "C-1001", "preferences")

store.put(namespace, "communication", {"channel": "email", "language": "bn"})
store.put(namespace, "tone", {"style": "concise"})

print("get one   :", store.get(namespace, "communication").value)
print("search all:")
for item in store.search(namespace):
    print("   ", item.key, "->", item.value)

In [ ]:
# Now prove a store crosses threads while checkpoint state does not.
from dataclasses import dataclass
from langgraph.runtime import Runtime


@dataclass
class Context:
    customer_id: str


class PrefState(TypedDict):
    question: str
    preferences: list


def load_preferences(state: PrefState, runtime: Runtime[Context]) -> dict:
    # runtime.store is injected by LangGraph; runtime.context carries the customer_id.
    ns = ("northstar", "customer", runtime.context.customer_id, "preferences")
    items = runtime.store.search(ns)
    return {"preferences": [item.value for item in items]}


pref_graph = (
    StateGraph(PrefState, context_schema=Context)
    .add_node("load_preferences", load_preferences)
    .add_edge(START, "load_preferences")
    .add_edge("load_preferences", END)
    .compile(checkpointer=InMemorySaver(), store=store)   # both kinds of memory
)

# Two DIFFERENT threads (tickets), SAME customer -> both see the stored prefs.
for ticket in ["ticket-1", "ticket-2"]:
    out = pref_graph.invoke(
        {"question": "hi"},
        {"configurable": {"thread_id": ticket}},
        context=Context(customer_id="C-1001"),
    )
    print(f"{ticket}: {out['preferences']}")

### 6.4 Checkpointer vs. store — how to choose, and a warning

Ask one question: **does this data belong to a single conversation, or to the customer?**

| You want to... | Use |
|---|---|
| Continue a conversation / remember what was said | Checkpointer (`thread_id`) |
| Pause for approval and resume later | Checkpointer |
| Inspect or replay how a decision was made | Checkpointer (state history) |
| Remember a preference across new tickets | Store (namespace) |
| Share knowledge between users/agents | Store |

**Memory is a product and privacy decision, not just a feature.** For anything you store, ask: did the user actually give us this fact? Do we need to keep it? Who may read this namespace? How is it corrected or deleted?

⚠️ **Production note:** `InMemorySaver` and `InMemoryStore` live in RAM and vanish when the process exits — they exist to teach the *semantics*. In production use a durable backend: `PostgresSaver` / `SqliteSaver` for checkpoints, `PostgresStore` / `RedisStore` for stores (or let the managed Agent Server handle it). Swapping them is a one-line change.

References: [Checkpointers](https://docs.langchain.com/oss/python/langgraph/checkpointers) · [Stores](https://docs.langchain.com/oss/python/langgraph/stores).


## 7. Recap

- A LangGraph graph is four things: **state** (shared typed data), **nodes** (functions that return updates), **edges** (what runs next), and **compile** (build it).
- **Conditional edges** let a small router function choose the next node — that is routing.
- **Workflows** fix the path in code; **agents** let the model choose; they are two ends of one spectrum, and `create_agent` is itself a LangGraph graph.
- **Multi-agent** systems split an overloaded agent into focused specialists — an agent can be wrapped as a tool for a supervisor.
- **Checkpointers** give one conversation durable memory, keyed by `thread_id`; you can inspect state and its full history.
- **Stores** hold facts across conversations, keyed by a namespace; use both, and treat stored memory as a privacy decision.
- In-memory persistence teaches the semantics; production uses a durable backend.

### Check yourself (no notes)

1. What are the four building blocks of a graph?
2. What does a node return, and why only the changed fields?
3. When would you enforce order with edges instead of trusting an agent's prompt?
4. What is the difference between a checkpointer and a store?
5. Why is `thread_id` not the same as a user id?

### References

- [Overview](https://docs.langchain.com/oss/python/langgraph/overview) · [Quickstart](https://docs.langchain.com/oss/python/langgraph/quickstart) · [Thinking in LangGraph](https://docs.langchain.com/oss/python/langgraph/thinking-in-langgraph) · [Workflows & agents](https://docs.langchain.com/oss/python/langgraph/workflows-agents)
- [Persistence](https://docs.langchain.com/oss/python/langgraph/persistence) · [Checkpointers](https://docs.langchain.com/oss/python/langgraph/checkpointers) · [Stores](https://docs.langchain.com/oss/python/langgraph/stores)
